In [ ]:
import pandas as pd
import numpy as np
import csv
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import os
import yfinance as yf
from transformers import pipeline
from huggingface_hub import login
import matplotlib.pyplot as plt
from datetime import datetime

load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)


In [ ]:
NewsAPI_data = pd.DataFrame(pd.read_csv('NewsAPI_data_multipleday.csv',encoding='utf-8'))
NewsAPI_data['type'] = 'news'
NewsAPI_data.drop_duplicates(subset=['title'],inplace=True)
NewsAPI_data['date'] = pd.to_datetime(NewsAPI_data['publishedAt']).dt.strftime('%y%m%d')
NewsAPI_data_needed = NewsAPI_data[['title','description','content','date']]

In [ ]:
prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
# finbert_model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone',num_labels=3)
# tokenizer_tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')

nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)
# nlp2 = pipeline('text-classification', model=finbert_model, tokenizer=tokenizer_tokenizer, truncation=True, max_length=512,batch_size=32)

def pos_neg_neu1(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1]
 
# def pos_neg_neu2(texts):
#     results2 = nlp2([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
#     return [ i['score'] if i['label'] == 'Positive' else -i['score'] if i['label'] == 'Negative' else 0 for i in results2]

NewsAPI_data_needed['title_score_prosus'] = pos_neg_neu1(NewsAPI_data_needed['title'])
NewsAPI_data_needed['description_score_prosus'] = pos_neg_neu1(NewsAPI_data_needed['description'])
NewsAPI_data_needed['content_score_prosus'] = pos_neg_neu1(NewsAPI_data_needed['content'])
NewsAPI_data_needed['final_score_prosus'] = NewsAPI_data_needed[['title_score_prosus', 'description_score_prosus','content_score_prosus']].mean(axis=1)

# NewsAPI_data_needed['title_score_yiyanghkust'] = pos_neg_neu2(NewsAPI_data_needed['title'])
# NewsAPI_data_needed['description_score_yiyanghkust'] = pos_neg_neu2(NewsAPI_data_needed['description'])
# NewsAPI_data_needed['final_score_yiyanghkust'] = NewsAPI_data_needed[['title_score_yiyanghkust', 'description_score_yiyanghkust']].mean(axis=1)

# NewsAPI_data_needed['score'] = NewsAPI_data_needed[['final_score_yiyanghkust','final_score_prosus']].mean(axis=1)

In [ ]:
NewsAPI_data_needed.reset_index(drop=True, inplace=True)
NewsAPI_data_needed.to_csv('NewsAPI_data_multipleday_withscores.csv', index=False, encoding='utf-8')


In [ ]:
temp = NewsAPI_data_needed.groupby('date')['final_score_prosus'].mean() 
plt.figure(figsize=(10, 5))
plt.plot(temp.index, temp.values, marker='o')


In [ ]:
temp

In [ ]:
oil_volatility_index = yf.download('^OVX', start='2026-02-28', end=datetime.today().date())
oil_volatility_index['avg'] = oil_volatility_index[['Open','Close']].mean(axis=1)
oil_volatility_index= oil_volatility_index[['avg']]

oil_volatility_index